In [ ]:
JOBS_GPU0 = []
JOBS_GPU1 = []
JOBS_CPU = []
RUN_TAG = 'vna_parallel_20261017'

import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
REPO_COMMIT = '2d4114b0bfa23e74847c23d15725515ac64397bc'
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000


# Parallel VNA pilot — validation hold
Run real T4 calibration after pytest, archive results, and stop. Full training awaits review of real schedules and baseline provenance.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_pilot_b3")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env={**test_env, "CUDA_VISIBLE_DEVICES": ""}, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "diagnostics/signal_curve.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO), str(GROMO_SRC), str(REFERENCE)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")

MU_REL_DEFAULT = 1e-2
OFFICIAL_MU = 1e-2
MU_SOURCE = "preregistered-pilot"


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.stage_b5 import restore_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs" / "stageB"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_archives(Path("/kaggle/input"), STAGE_B)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/stageB",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from experiments.parallel_validity import validate_baseline, validate_record
ROOT = STAGE_B / RUN_TAG
ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")

def completed(job):
    path = ROOT/job/"summary.json"
    if not path.exists(): return False
    summary = json.loads(path.read_text())
    if summary.get("status") != "COMPLETED": return False
    protocol_path = ROOT/job/"protocol.json"
    protocol = json.loads(protocol_path.read_text())
    tasks = json.loads((ROOT/job/"tasks.json").read_text())
    validate_record(job, summary, protocol, tasks, REPO_COMMIT)
    return True

def cil_command(job):
    match = re.fullmatch(r"(icarl(?:_vna(?:_random|_euclid)?|_ngd)?)__(inc5|inc10|b50inc10)__s(\d+)", job)
    if not match: raise ValueError("unknown CIL job "+job)
    label, protocol, seed = match.groups()
    arm = "icarl_ngd" if label == "icarl_ngd" else "icarl_vna" if label.startswith("icarl_vna") else "icarl"
    selection = "random_frob" if label.endswith("_random") else "tiny_V_euclid" if label.endswith("_euclid") else "tiny_V"
    init_cls, increment = (50,10) if protocol == "b50inc10" else (10,5 if protocol == "inc5" else 10)
    cmd = [sys.executable,"-u","-m","experiments.pilot_b3","--arm",arm,
           "--selection",selection,"--seed",seed,"--init-cls",str(init_cls),
           "--increment",str(increment),"--out",str(ROOT/job),"--device","cuda:0",
           "--workers","2","--data-dir",str(DATA_ROOT),"--no-measure"]
    if arm != "icarl": cmd += ["--auto-schedule"]
    return cmd

def b5_command(job):
    match = re.fullmatch(r"b5_(.+)__s([012])", job)
    if not match: raise ValueError("unknown B5 job "+job)
    label, seed = match.groups()
    arms = {"bp":"bp", "cbp":"cbp", "cbp_gir":"cbp_gir",
            "cbp_gir_randdir":"cbp_gir_randdir", "cbp_gir_vna":"cbp_gir_vna_l1",
            "bp_vna":"bp_vna_l1", "bp_ngd":"bp_ngd_l1"}
    return [sys.executable,"-u","-m","experiments.pilot_b5","--arm",arms[label],
            "--seed",seed,"--out",str(ROOT/job),"--flat-output","--resume"]

def run_gpu(gpu, jobs):
    for job in jobs:
        if pause_due(): break
        if completed(job): continue
        if not launch(cil_command(job), gpu, job, RUN_ROOT/f"{RUN_TAG}_{job}.log"): break
        archive_outputs()

def run_cpu(job):
    if pause_due() or completed(job): return
    from experiments.vna_resume import launch_logged
    return launch_logged(b5_command(job), cwd=REPO,
        env={**ENV,"CUDA_VISIBLE_DEVICES":""}, log_path=RUN_ROOT/f"{RUN_TAG}_{job}.log",
        lane="CPU", job=job, lanes=lane_state, paused=session_paused, deadline=VNA_DEADLINE)

def reuse_existing_baseline():
    target = ROOT/"icarl__inc10__s1993"
    if target.exists():
        validate_baseline(target, REPO_COMMIT, DATA_ROOT)
        return
    for path in Path("/kaggle/input").rglob("summary.json"):
        try: summary = json.loads(path.read_text())
        except (OSError,ValueError): continue
        if summary.get("arm") == "icarl" and summary.get("seed") == 1993:
            try: validate_baseline(path.parent, REPO_COMMIT, DATA_ROOT)
            except (OSError, ValueError, KeyError) as error:
                print("Rejected attached baseline:", path, str(error))
                continue
            target.mkdir(parents=True, exist_ok=True)
            for name in ("summary.json","tasks.json","protocol.json","reproduction_gate.json","vna_log.csv"):
                source = path.parent/name
                if source.exists(): shutil.copy2(source,target/name)
            print("Reused attached inc10 seed1993 baseline:",path)
            return
    raise RuntimeError("No validated seed1993 baseline: attach matching dataset/protocol/commit artifacts")

def real_t4_preflight(gpu, jobs):
    for job in jobs:
        cmd = cil_command(job)
        cmd[cmd.index("--out")+1] = str(ROOT/"_preflight"/job)
        if "--auto-schedule" in cmd: cmd.remove("--auto-schedule")
        cmd += ["--smoke"]
        if "icarl_vna" in job or "icarl_ngd" in job: cmd += ["--calibrate"]
        if not launch(cmd, gpu, job+"_preflight", RUN_ROOT/f"{RUN_TAG}_{job}_preflight.log"):
            return

try:
    # Until actual T4 calibration has been reviewed, these artifacts are
    # smoke-only. No CPU or GPU full-training job may start in this revision.
    import torch
    assert torch.cuda.device_count() == 2
    assert all("T4" in torch.cuda.get_device_name(i) for i in range(2)), "T4 x2 required"
    with ThreadPoolExecutor(max_workers=2) as preflight_pool:
        list(preflight_pool.map(lambda pair: real_t4_preflight(*pair),
                                enumerate((JOBS_GPU0,JOBS_GPU1))))
    archive_outputs()
    raise RuntimeError("FULL RUN HELD: review archived real T4 smoke schedules before enabling full training")
    reuse_existing_baseline()
    with ThreadPoolExecutor(max_workers=2) as gpu_pool, ThreadPoolExecutor(max_workers=4) as cpu_pool:
        futures = [gpu_pool.submit(run_gpu,0,JOBS_GPU0), gpu_pool.submit(run_gpu,1,JOBS_GPU1)]
        futures += [cpu_pool.submit(run_cpu,job) for job in JOBS_CPU]
        for future in futures: future.result()
finally:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs()

rows=[]
for path in sorted(ROOT.glob("*/summary.json")):
    summary=json.loads(path.read_text()); job=path.parent.name
    if summary.get("status") != "COMPLETED": continue
    tasks_path=path.parent/"tasks.json"
    tasks=json.loads(tasks_path.read_text()) if tasks_path.exists() else summary.get("tasks",[])
    if job.startswith("icarl") and tasks:
        last=tasks[-1]
        rows.append(dict(job=job, cnn_average=last.get("cnn_average_accuracy"),
            nme_average=last.get("nme_average_accuracy"), cnn_last=last.get("cnn_top1"),
            nme_last=last.get("nme_top1"), cnn_forgetting=last.get("cnn_forgetting"),
            nme_forgetting=last.get("nme_forgetting")))
    elif job.startswith("b5_"):
        rows.append(dict(job=job, mean_last50=summary.get("mean_last50"), variants=len(summary.get("variants",[]))))
import pandas as pd
display(pd.DataFrame(rows))
if session_paused.is_set(): print("PAUSED: attach this output and rerun to resume")
if archive_errors: raise RuntimeError(archive_errors)
